# Census Trade Data Download (Google Colab)

Downloads monthly U.S. Census imports by port × HS4 × country × month for 2000–2026
and saves one Parquet file per year to Google Drive.

**Speed**: uses `ThreadPoolExecutor` to fire 10 concurrent API requests,
giving ~10× speedup over sequential downloads.

**Before running:**
1. Add your Census API key via *Secrets* (key icon in the left panel) → name: `CENSUS_API_KEY`
2. Run all cells top to bottom; already-saved years are skipped automatically.

In [ ]:
# Install polars (not pre-installed in Colab)
%pip install -q polars

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Output directory on your Drive
import pathlib
OUT_DIR = pathlib.Path('/content/drive/MyDrive/census_trade_data')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')

In [ ]:
# Read API key from Colab Secrets (add via the key icon in the left sidebar)
CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')
if not CENSUS_API_KEY:
    raise RuntimeError('Add CENSUS_API_KEY to Colab Secrets before running.')
print('API key loaded.')

In [ ]:

from __future__ import annotations

import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import polars as pl

CENSUS_URL = 'https://api.census.gov/data/timeseries/intltrade/imports/porths'
CENSUS_FIELDS = (
    'YEAR', 'MONTH', 'PORT', 'PORT_NAME',
    'CTY_CODE', 'CTY_NAME',
    'I_COMMODITY', 'I_COMMODITY_LDESC', 'COMM_LVL', 'SUMMARY_LVL',
    'GEN_VAL_MO', 'VES_VAL_MO', 'VES_WGT_MO', 'CNT_VAL_MO', 'CNT_WGT_MO',
)
INTEGER_FIELDS = ('GEN_VAL_MO', 'VES_VAL_MO', 'VES_WGT_MO', 'CNT_VAL_MO', 'CNT_WGT_MO')
HS_PREFIXES = tuple(f'{digit}*' for digit in range(10))
USER_AGENT = 'supply-chain-resilience/0.1 (colab data fetcher)'


def _ts() -> str:
    return datetime.now().strftime('%H:%M:%S')


def _get_bytes(url: str, params: dict, timeout: int = 120) -> bytes:
    full_url = f'{url}?{urlencode(params, doseq=True)}'
    req = Request(full_url, headers={'User-Agent': USER_AGENT})
    for attempt in range(4):
        try:
            with urlopen(req, timeout=timeout) as r:
                return r.read()
        except HTTPError as e:
            detail = e.read().decode('utf-8', errors='replace')[:300]
            if e.code in {429, 500, 502, 503, 504} and attempt < 3:
                wait = 2 ** attempt
                print(f'  [{_ts()}] HTTP {e.code}, retry {attempt+1}/4 in {wait}s')
                time.sleep(wait)
            else:
                raise RuntimeError(f'HTTP {e.code}: {detail}') from e
        except (TimeoutError, OSError, URLError) as e:
            if attempt < 3:
                wait = 2 ** attempt
                print(f'  [{_ts()}] network error, retry {attempt+1}/4 in {wait}s — {e}')
                time.sleep(wait)
            else:
                raise RuntimeError(f'Network error after 4 attempts: {e}') from e
    raise RuntimeError('Unreachable')


def _dedup(cols: list[str], rows: list[list]) -> tuple[list[str], list[list]]:
    seen: set[str] = set()
    keep = [i for i, c in enumerate(cols) if not (c in seen or seen.add(c))]
    return [cols[i] for i in keep], [[r[i] for i in keep] for r in rows]


def fetch_one(month: str, port: str, hs_prefix: str, api_key: str) -> pl.DataFrame:
    """Fetch HS4 × country rows for one port × month × first-digit prefix.

    Census recommends splitting large country-by-commodity calls with wildcard
    classification prefixes. SUMMARY_LVL='DET' returns individual trading partners
    and the CTY_CODE='-' all-country total; both are retained.
    """
    params = {
        'get': ','.join(CENSUS_FIELDS),
        'time': month,
        'PORT': port,
        'I_COMMODITY': hs_prefix,
        'COMM_LVL': 'HS4',
        'SUMMARY_LVL': 'DET',
        'key': api_key,
    }
    raw = _get_bytes(CENSUS_URL, params)
    if not raw.strip():
        return pl.DataFrame()
    payload = json.loads(raw)
    if isinstance(payload, dict) or not payload:
        return pl.DataFrame()
    cols, *rows = payload
    cols, rows = _dedup(cols, rows)
    frame = pl.DataFrame(rows, schema=cols, orient='row')
    if 'time' not in frame.columns:
        frame = frame.with_columns(pl.lit(month).alias('time'))
    num_cols = [c for c in INTEGER_FIELDS if c in frame.columns]
    if num_cols:
        frame = frame.with_columns([pl.col(c).cast(pl.Int64, strict=False) for c in num_cols])
    # Defensive checks: the predicates should already restrict the response to HS4/DET.
    if 'COMM_LVL' in frame.columns:
        frame = frame.filter(pl.col('COMM_LVL') == 'HS4')
    if 'SUMMARY_LVL' in frame.columns:
        frame = frame.filter(pl.col('SUMMARY_LVL') == 'DET')
    return frame


def iter_months(start: str, end: str) -> list[str]:
    s = date.fromisoformat(f'{start}-01')
    e = date.fromisoformat(f'{end}-01')
    result, cur = [], s
    while cur <= e:
        result.append(cur.strftime('%Y-%m'))
        cur = date(cur.year + (cur.month == 12), cur.month % 12 + 1, 1)
    return result


def download_year_concurrent(
    year: int,
    ports: list[str],
    api_key: str,
    max_workers: int = 10,
) -> pl.DataFrame:
    end_month = f'{year}-{date.today().month:02d}' if year == date.today().year else f'{year}-12'
    months = iter_months(f'{year}-01', end_month)

    # Ten wildcard calls per port-month keep country × HS4 responses below API limits.
    tasks = [(m, p, prefix) for m in months for p in ports for prefix in HS_PREFIXES]
    total = len(tasks)
    print(f'  [{_ts()}] Submitting {total:,} tasks ({len(months)} months × {len(ports)} ports '
          f'× {len(HS_PREFIXES)} HS prefixes) '
          f'to {max_workers} workers...')

    frames: list[pl.DataFrame] = []
    n_err = 0
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(fetch_one, m, p, prefix, api_key): (m, p, prefix)
            for m, p, prefix in tasks
        }
        for i, future in enumerate(as_completed(futures), 1):
            try:
                result = future.result()
                if not result.is_empty():
                    frames.append(result)
            except RuntimeError as e:
                n_err += 1
                if n_err <= 5:
                    m, p, prefix = futures[future]
                    print(f'  [{_ts()}] ✗ {m} port={p} prefix={prefix}: {e}')

            if i % 50 == 0 or i == total:
                elapsed = time.time() - t0
                rate = i / elapsed if elapsed > 0 else 0
                eta_s = (total - i) / rate if rate > 0 else 0
                rows_so_far = sum(len(f) for f in frames)
                print(
                    f'  [{_ts()}] {i:,}/{total:,} done  '
                    f'({i/total*100:.1f}%)  '
                    f'{rate:.1f} req/s  '
                    f'ETA {eta_s/60:.1f} min  '
                    f'{rows_so_far:,} rows  '
                    f'{n_err} errors'
                )

    print(f'  [{_ts()}] Finished: {total - n_err:,} ok, {n_err} errors, '
          f'{sum(len(f) for f in frames):,} total rows')
    if n_err:
        raise RuntimeError(
            f'Year is incomplete: {n_err} API tasks failed. Rerun before saving.'
        )
    return pl.concat(frames, how='diagonal_relaxed') if frames else pl.DataFrame()

print(f'[{_ts()}] Functions defined.')


In [ ]:

# U.S. seaport codes verified against Census Schedule D.
# Reference: https://www.census.gov/foreign-trade/schedules/d/distcode.html
#
# CORE_PORTS are the original major gateways. REGIONAL_ALTERNATIVE_PORTS adds
# smaller and medium-sized seaports that could absorb cargo after a disruption.
# Selection as an actual alternative must still be done ex ante by HS4, source
# country, pre-event activity, and whether the candidate was unaffected by the event.
CORE_PORTS = [
    # Pacific Coast major gateways
    '2704',  # Los Angeles, CA
    '2709',  # Long Beach, CA
    '2811',  # Oakland, CA
    '2904',  # Portland, OR
    '3001',  # Seattle, WA
    '3002',  # Tacoma, WA
    # Atlantic Coast major gateways
    '1001',  # New York, NY
    '1003',  # Newark, NJ
    '1101',  # Philadelphia, PA
    '1103',  # Wilmington, DE
    '1303',  # Baltimore, MD
    '1401',  # Norfolk-Newport News, VA
    '1601',  # Charleston, SC
    '1703',  # Savannah, GA
    '1801',  # Tampa, FL
    '1803',  # Jacksonville, FL
    # South Florida
    '5201',  # Miami, FL
    '5203',  # Port Everglades, FL
    # Gulf Coast
    '1901',  # Mobile, AL
    '2002',  # New Orleans, LA
    '5301',  # Houston, TX
    '5310',  # Galveston, TX
    # Hawaii & Puerto Rico
    '3201',  # Honolulu, HI
    '4909',  # San Juan, PR
]

REGIONAL_ALTERNATIVE_PORTS = [
    # Pacific regional alternatives / controls
    '2501',  # San Diego, CA
    '2713',  # Port Hueneme, CA
    '2812',  # Richmond, CA
    '2905',  # Longview, WA
    '2908',  # Vancouver, WA
    # Northeast regional alternatives / controls
    '0401',  # Boston, MA
    '0502',  # Providence, RI
    # Southeast Atlantic alternatives
    '1501',  # Wilmington, NC
    '1511',  # Beaufort-Morehead City, NC
    '1701',  # Brunswick, GA
    '1805',  # Fernandina Beach, FL
    # Florida Atlantic and Gulf alternatives
    '1816',  # Port Canaveral, FL
    '1818',  # Panama City, FL
    '1819',  # Pensacola, FL
    '1821',  # Port Manatee, FL
    '5204',  # West Palm Beach, FL
    # Alabama and Mississippi Gulf alternatives
    '1902',  # Gulfport, MS
    '1903',  # Pascagoula, MS
    # Lower Mississippi and southwest Louisiana alternatives
    '2004',  # Baton Rouge, LA
    '2010',  # Gramercy, LA (Lower Mississippi terminals)
    '2017',  # Lake Charles, LA
    # Sabine-Neches alternatives
    '2101',  # Port Arthur, TX
    '2102',  # Sabine, TX
    '2103',  # Orange, TX
    '2104',  # Beaumont, TX
    # Texas Gulf alternatives
    '5306',  # Texas City, TX
    '5311',  # Freeport, TX
    '5312',  # Corpus Christi, TX
    '5313',  # Port Lavaca, TX
]

# Keep the existing variable name for downstream cells. The expanded list is the
# default because Census totals are cheap to collect relative to raw AIS data.
MAJOR_PORTS = CORE_PORTS + REGIONAL_ALTERNATIVE_PORTS
assert len(MAJOR_PORTS) == len(set(MAJOR_PORTS)), 'Duplicate Schedule D port code'
assert all(len(p) == 4 and p.isdigit() for p in MAJOR_PORTS)

# Version output names by port universe so previously saved 24-port files do not
# incorrectly cause the expanded downloads to be skipped.
PORT_SET_TAG = f'expanded_{len(MAJOR_PORTS)}ports'
DATA_GRAIN_TAG = 'country'

# Ten wildcard requests per port-month return HS4 × individual-country detail.
REQUESTS_PER_YEAR = len(MAJOR_PORTS) * 12 * len(HS_PREFIXES)
print(f'[{_ts()}] {len(CORE_PORTS)} core + {len(REGIONAL_ALTERNATIVE_PORTS)} regional '
      f'= {len(MAJOR_PORTS)} ports')
print(f'[{_ts()}] {REQUESTS_PER_YEAR:,} requests per full year '
      f'({len(HS_PREFIXES)} HS-prefix calls per port-month)')


In [ ]:

# ── Configure the download range ──────────────────────────────────────────────
START_YEAR  = 2013   # adjust as needed
END_YEAR    = 2025
MAX_WORKERS = 10     # concurrent API requests; increase to 20 if no 429 errors
# ──────────────────────────────────────────────────────────────────────────────

years_to_run = [y for y in range(START_YEAR, END_YEAR + 1)
                if not (OUT_DIR / f'census_imports_porths_hs4_{DATA_GRAIN_TAG}_{PORT_SET_TAG}_{y}_retrieved_{date.today():%Y%m%d}.parquet').exists()]
years_done   = (END_YEAR - START_YEAR + 1) - len(years_to_run)

print(f'[{_ts()}] Download plan: {START_YEAR}–{END_YEAR}')
print(f'  Already saved: {years_done} year(s)')
print(f'  To download:   {len(years_to_run)} year(s) → {years_to_run}')
print(f'  Max workers:   {MAX_WORKERS}')
print()

session_t0 = time.time()

for year_idx, year in enumerate(years_to_run, 1):
    out_path = OUT_DIR / f'census_imports_porths_hs4_{DATA_GRAIN_TAG}_{PORT_SET_TAG}_{year}_retrieved_{date.today():%Y%m%d}.parquet'

    print(f'[{_ts()}] ── Year {year}  ({year_idx}/{len(years_to_run)}) ──────────────────')

    year_t0 = time.time()
    df = download_year_concurrent(year, MAJOR_PORTS, CENSUS_API_KEY, MAX_WORKERS)
    year_elapsed = time.time() - year_t0

    if not df.is_empty():
        n_country_rows = df.filter(pl.col('CTY_CODE') != '-').height
        n_total_rows = df.filter(pl.col('CTY_CODE') == '-').height
        if n_country_rows == 0:
            raise RuntimeError(
                f'{year} returned no individual-country rows; refusing to save totals-only data.'
            )
        print(f'  [{_ts()}] Country-detail rows: {n_country_rows:,}; '
              f'all-country total rows: {n_total_rows:,}')
        print(f'  [{_ts()}] Writing {len(df):,} rows to Drive...', end=' ', flush=True)
        df.write_parquet(out_path)
        print('done.')
        print(f'  [{_ts()}] ✓ Saved {year}: {out_path.name}  ({year_elapsed/60:.1f} min)')
    else:
        print(f'  [{_ts()}] ✗ No data returned for {year}')

    session_elapsed = time.time() - session_t0
    avg_per_year = session_elapsed / year_idx
    remaining_years = len(years_to_run) - year_idx
    print(f'  [{_ts()}] Session: {session_elapsed/60:.1f} min elapsed, '
          f'~{avg_per_year * remaining_years / 60:.1f} min remaining for {remaining_years} year(s)')
    print()

total_elapsed = time.time() - session_t0
print(f'[{_ts()}] All done. Total session time: {total_elapsed/60:.1f} min')
